In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge
from sklearn.metrics import classification_report, mean_squared_error

In [ ]:
# ==========================================
# PART 1: BREAST CANCER (LOGISTIC REGRESSION)
# ==========================================

In [ ]:
# 1. Load the dataset
bc_data = load_breast_cancer()
X_bc = bc_data.data
y_bc = bc_data.target  # 0 = malignant, 1 = benign

In [ ]:
# 2. Split the dataset (80:20, random_state=9001)
X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.20, random_state=9001
)

In [ ]:
# 3. Scale the data using MinMaxScaler
scaler_bc = MinMaxScaler()
# Fit ONLY on the training data, then transform both
X_bc_train_scaled = scaler_bc.fit_transform(X_bc_train)
X_bc_test_scaled = scaler_bc.transform(X_bc_test)

In [ ]:
# 4. Initialize and fit Logistic Regression (default solver)
log_reg = LogisticRegression()
log_reg.fit(X_bc_train_scaled, y_bc_train)

In [ ]:
# 5. Predict and generate classification report
y_bc_train_pred = log_reg.predict(X_bc_train_scaled)
y_bc_test_pred = log_reg.predict(X_bc_test_scaled)

print("\nClassification Report (Training Data):")
print(classification_report(y_bc_train, y_bc_train_pred, digits=4))

print("\nClassification Report (Testing Data):")
print(classification_report(y_bc_test, y_bc_test_pred, digits=4))

In [ ]:
# ==========================================
# PART 2: DIABETES (LINEAR, L1, L2 REGRESSION)
# ==========================================

In [ ]:
# 1. Load the dataset
diab_data = load_diabetes()
X_db = diab_data.data
y_db = diab_data.target # Disease progression score
feature_names = diab_data.feature_names

In [ ]:
# 2. Split the dataset (80:20, random_state=9001)
X_db_train, X_db_test, y_db_train, y_db_test = train_test_split(
    X_db, y_db, test_size=0.20, random_state=9001
)

In [ ]:
# 3. Scale the data using MinMaxScaler
scaler_db = MinMaxScaler()
# Fit ONLY on the training data, then transform both
X_db_train_scaled = scaler_db.fit_transform(X_db_train)
X_db_test_scaled = scaler_db.transform(X_db_test)

In [ ]:
# 4. Initialize and fit the three models
# Model A: No Regularization (Standard Linear Regression)
lr = LinearRegression()
lr.fit(X_db_train_scaled, y_db_train)

# Model B: L1 Regularization (Lasso) with alpha = 1
lasso = Lasso(alpha=1.0)
lasso.fit(X_db_train_scaled, y_db_train)

# Model C: L2 Regularization (Ridge) with alpha = 1
ridge = Ridge(alpha=1.0)
ridge.fit(X_db_train_scaled, y_db_train)

In [ ]:
# 5. Obtain Mean Squared Error (MSE) for all three models
mse_test = {
    'Linear (No Reg)': mean_squared_error(y_db_test, lr.predict(X_db_test_scaled)),
    'Lasso (L1)': mean_squared_error(y_db_test, lasso.predict(X_db_test_scaled)),
    'Ridge (L2)': mean_squared_error(y_db_test, ridge.predict(X_db_test_scaled))
}

print("\nMean Squared Errors (Testing Data):")
for model_name, mse in mse_test.items():
    print(f"{model_name}: {mse:.2f}")

In [ ]:
# 6. Obtain coefficients
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Linear': lr.coef_,
    'Lasso_L1': lasso.coef_,
    'Ridge_L2': ridge.coef_
})

print("\nModel Coefficients:")
print(coef_df.round(4).to_string(index=False))

In [ ]:
# Graded questions below

#Question 1: What is the accuracy obtained from binary classification? Round to 2 dec places.
#Answer: 1.00

from sklearn.metrics import accuracy_score

# Calculate accuracy on the test set
test_accuracy = accuracy_score(y_bc_test, y_bc_test_pred)
print(f"Test Set Accuracy: {test_accuracy:.2f}")

# Calculate accuracy on the training set (just in case)
train_accuracy = accuracy_score(y_bc_train, y_bc_train_pred)
print(f"Training Set Accuracy: {train_accuracy:.2f}")

In [ ]:
#Question 2: In breast cancer logistic regression model, assume we obtained a coefficient of X for the feature ‘symmetry error’. What does this mean?
#Answer: One unit increase in symmetry error changes the log odds of the tumour being benign by X.

#Explanation: In Logistic Regression, the algorithm does not predict probabilities directly; it predicts the log-odds. Therefore, the raw coefficient exactly represents the change in log-odds for a 1-unit increase in the feature. Because '1' maps to benign in this dataset, it predicts the log-odds of the tumor being benign.

In [ ]:
#Question 3: Which feature has the highest magnitude coefficient? (Mean fractal dimension, Area error, Mean perimeter, Worst radius)
#Answer: Worst radius

import numpy as np

# Extract all coefficients from the logistic regression model
coefs = log_reg.coef_[0]

# Create a dataframe with the features and their absolute magnitudes
coef_df_bc = pd.DataFrame({
    'Feature': bc_data.feature_names, 
    'Coefficient': coefs
})
coef_df_bc['Magnitude'] = np.abs(coef_df_bc['Coefficient'])

# Filter for the 4 options provided in the question
options = ['mean fractal dimension', 'area error', 'mean perimeter', 'worst radius']
filtered_df = coef_df_bc[coef_df_bc['Feature'].isin(options)].sort_values(by='Magnitude', ascending=False)

print("Coefficients ranked by magnitude:")
print(filtered_df[['Feature', 'Magnitude']])

In [ ]:
#Question 4: Which model has lowest mean squared error?
#Answer: Model with no regularisation (Standard Linear Regression)

# To verify Question 4 programmatically:
print("Mean Squared Errors on Testing Data:")
for model_name, mse in mse_test.items():
    print(f"{model_name}: {mse:.2f}")

# Find the lowest programmatically
best_model = min(mse_test, key=mse_test.get)
print(f"\nModel with the lowest MSE: {best_model}")

In [ ]:
#Question 5: How many features has L1 regularisation eliminated, considering coefficients equal to zero at a precision of 3 decimal places?
#Answer: 4

# To verify Question 5 programmatically:

# Round the Lasso (L1) coefficients to 3 decimal places
lasso_coefs_rounded = np.round(lasso.coef_, 3)

# Count how many are exactly 0.000
eliminated_count = np.sum(lasso_coefs_rounded == 0.000)

print(f"Number of features eliminated by L1 (Lasso): {eliminated_count}")

# (Optional) Show which specific features were eliminated
eliminated_features = [feature_names[i] for i in range(len(lasso_coefs_rounded)) if lasso_coefs_rounded[i] == 0.000]
print(f"The eliminated features are: {eliminated_features}")